In [6]:
import pandas as pd

data = pd.read_csv("regression_data.csv")

data.head()
list(data)

['timestamp_utc',
 'wind_actual_mw',
 'da_price_eur_mwh',
 'wind_forecast_mw',
 'ida2_price_eur_mwh',
 'delivery_date',
 'delivery_week_start',
 'timestamp_local',
 'interval_end_utc',
 'delivery_period_index',
 'hour_of_day',
 'quarter_hour_of_day',
 'day_of_week',
 'month',
 'year',
 'is_weekend',
 'post_da_15min',
 'ida2_gate_close_utc',
 'auction_to_delivery_hours',
 'hydro_current_week_mwh',
 'assumed_available_utc',
 'hydro_observation_week_start',
 'hydro_storage_mwh',
 'hydro_storage_gwh',
 'spread_eur_mwh',
 'wind_error_mw',
 'wind_error_gw',
 'wind_surplus_gw',
 'wind_shortfall_gw',
 'hydro_centered_gwh',
 'wind_surplus_hydro_interaction',
 'wind_shortfall_hydro_interaction',
 'gap_minutes_from_previous_row',
 'new_contiguous_segment',
 'contiguous_segment_id',
 'da_timestamp_repaired']

In [10]:
import pandas as pd
import statsmodels.formula.api as smf

data["spread"] = data["da_price_eur_mwh"] - data["ida2_price_eur_mwh"]

print(data[["wind_surplus_gw", "wind_shortfall_gw"]].describe().loc[["min", "max"]])

data = data.sort_values("timestamp_utc").dropna(
    subset=["spread", "wind_surplus_gw", "wind_shortfall_gw"]
)

model = smf.ols(
    "spread ~ wind_surplus_gw + wind_shortfall_gw ",
    data=data,
).fit(cov_type="HAC", cov_kwds={"maxlags": 96})   # Newey-West, 96 = one day of 15-min steps

print(model.summary().tables[1])

# asymmetry test (use the line that matches your shortfall sign)
# shortfall stored as positive magnitude -> symmetric means b1 = -b2:
print(model.t_test("wind_surplus_gw + wind_shortfall_gw = 0"))
# shortfall stored as negative number -> symmetric means b1 = b2:
# print(model.t_test("wind_surplus_gw - wind_shortfall_gw = 0"))

     wind_surplus_gw  wind_shortfall_gw
min            0.000              0.000
max            2.128              1.632
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            -1.0426      0.360     -2.898      0.004      -1.748      -0.337
wind_surplus_gw       2.6513      1.368      1.937      0.053      -0.031       5.333
wind_shortfall_gw     2.5946      1.669      1.555      0.120      -0.676       5.865
                             Test for Constraints                             
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
c0             5.2458      2.495      2.103      0.035       0.356      10.136


In [ ]:
model = smf.ols(
    "spread ~ wind_surplus_gw + wind_shortfall_gw ",
    data=data,
).fit(cov_type="HAC", cov_kwds={"maxlags": 96})   # Newey-West, 96 = one day of 15-min steps

print(model.summary().tables[1])